## 6. DeltaSharing

Publica las tablas gold del catalogo indicado en un share de Delta Sharing y lo
asigna al recipient que consume Power BI. Se puede ejecutar varias veces.

- `1.Preparacion_Ambiente` borra y recrea el catalogo, asi que las tablas del share
  quedan apuntando a objetos que ya no existen. Por eso se quitan y se vuelven a agregar.
- Power BI no lee tablas con deletion vectors: se desactivan y se reescriben los archivos
  antes de compartir, y se comparten sin historial.

In [0]:
CREATE WIDGET TEXT catalogo DEFAULT "football_prod";
CREATE WIDGET TEXT share_name DEFAULT "FOOTBALL_SHARE";
CREATE WIDGET TEXT recipient_name DEFAULT "ExternalCompanys";

In [0]:
SELECT * FROM $catalogo.gold.season_stats LIMIT 10

In [0]:
CREATE SHARE IF NOT EXISTS $share_name

In [0]:
%python
catalogo = dbutils.widgets.get("catalogo")
share_name = dbutils.widgets.get("share_name")
tablas = ["season_stats", "standings", "calendar_stats", "club_elo", "match_predictions"]

for tabla in tablas:
    nombre = f"{catalogo}.gold.{tabla}"
    spark.sql(f"ALTER TABLE {nombre} SET TBLPROPERTIES ('delta.enableDeletionVectors' = 'false')")
    spark.sql(f"REORG TABLE {nombre} APPLY (PURGE)")

for r in spark.sql(f"SHOW ALL IN SHARE {share_name}").collect():
    d = r.asDict()
    if d.get("type") == "TABLE":
        spark.sql(f"ALTER SHARE {share_name} REMOVE TABLE {d['name']}")
        print(f"{d['name']} quitada de {share_name}")

for tabla in tablas:
    nombre = f"{catalogo}.gold.{tabla}"
    spark.sql(f"ALTER SHARE {share_name} ADD TABLE {nombre} AS gold.{tabla}")
    print(f"{nombre} agregada a {share_name} como gold.{tabla}")

In [0]:
CREATE RECIPIENT IF NOT EXISTS $recipient_name

In [0]:
GRANT SELECT ON SHARE $share_name TO RECIPIENT $recipient_name

In [0]:
SHOW ALL IN SHARE $share_name